# A maintained Cartesian diopter: numerical evidence

This notebook analyzes completed simulations; it does **not** run an optimizer or a fluid solver.
The 1 MHz excitation forms a near-Cartesian mean surface and recovers from two disturbances
with every physical row drive fixed. The claim is limited to the declared axisymmetric,
isothermal model and tested trajectories.

The [research report](../docs/numerical-stability.md) records equations, sources, failed earlier
candidates and omitted physics. The [reproduction guide](../docs/stable-cartesian-reproduction.md)
contains the simulation commands. The viewer starts at its final frame, **0.2 s**; restart/play
reveals the computed formation trajectory.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from acoustic_freeform.single_interface.config import LensConfig
from acoustic_freeform.single_interface.surface import SurfaceSpace
from acoustic_freeform.single_interface.optics import trace_surface

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "Run from the repository or notebooks directory."
campaign = root / "artifacts/studies/S01-single-interface/cartesian-maintenance-2026-09-06"
run = campaign / "step-formation-dt00125"
report = json.loads((run / "report.json").read_text())
audit = json.loads((campaign / "verification/maintenance.json").read_text())
data = np.load(run / "trajectory.npz")
cfg = LensConfig(**report["configuration"])
space = SurfaceSpace(cfg)
print(report["control"])
print("Configuration:", cfg.name, "| frequency:", cfg.frequency_hz / 1e6, "MHz")


## 1. What optical surface is requested?

Let $r$ be radius, $z(r)$ the height relative to the target vertex, $n_o$ and $n_i$ the incident
and transmitted optical indices, and $z_o<0$, $z_i>0$ the real object and image coordinates.
For this real-conjugate case the unsquared Fermat equation is

\[
n_o\sqrt{r^2+[z(r)-z_o]^2}+n_i\sqrt{r^2+[z_i-z(r)]^2}
=n_o|z_o|+n_i z_i,\qquad z(0)=0.
\]

The numerical target uses $n_o=1.52$, $z_o=-50\,\mathrm{mm}$, $n_i=1$, $z_i=20\,\mathrm{mm}$.
Mounting shifts the graph to meet the 4 mm radius rim. The optical pupil has radius 3 mm.
The incident spherical wavefront is prescribed **inside** the liquid; exterior illumination
through the base window is not designed here.

The signed general construction and independent parameterization come from
[Silva-Lora and Torres (2020), JOSA A](https://doi.org/10.1364/JOSAA.392795).
Axial stigmatism does not by itself establish achromatism or off-axis aplanatism.


In [ ]:
radius = np.linspace(0, cfg.radius_m, 601)
fig, ax = plt.subplots(figsize=(8, 4))
for c, label, style in [(data["initial"], "Unforced liquid", "-"),
                         (data["target"], "Cartesian target", "--"),
                         (data["coefficients"][-1], "Computed held surface", "-")]:
    ax.plot(radius * 1000, cfg.radius_m * space.evaluate(c, radius / cfg.radius_m) * 1000,
            style, label=label)
ax.axvline(cfg.clear_radius_m * 1000, color="gray", ls=":", label="Clear pupil edge")
ax.set(xlabel="Radius (mm)", ylabel="Height above rim (mm)", title="Equal liquid volume; actual forward solution")
ax.legend(); ax.grid(alpha=.2)
plt.show()
print(f"Liquid volume: {report['liquid_volume_m3'] * 1e9:.3f} µL")
print(f"Actual vertex: {cfg.radius_m * space.evaluate(data['coefficients'][-1], 0) * 1000:.6f} mm")


## 2. How are the emitter excitations obtained?

Let $a$ denote volume-constrained surface coefficients and $w\in\mathbb C^{16}$ the complex
peak normal velocities of the array rows. A row uses a prescribed axial cosine taper and
all its azimuthal sectors share the same excitation. The acoustic boundary-value problem
is linear in $w$ on a fixed geometry. Consequently each generalized radiation load has the
form $f_j(a,w)=w^\dagger H_j(a)w$, with Hermitian matrix $H_j$ retaining coherent cross terms.

Bulk absorption creates a distributed mean-flow force. With $C$ the discrete kinematic map,
$A^{-1}$ the constrained incompressible Stokes inverse, and $b_E$ the bulk force vector,
its stationary equivalent shape load is

\[
f_E^{\rm eq}=(CA^{-1}C^T)^{-1}CA^{-1}b_E.
\]

This equivalent is used only in the stationary inverse. Physical trajectories apply $b_E$
as a **distributed force** and retain inertia and convection. Array fitting combines local
ray/height error, integrated pressure and drive regularization, then alternates with the
nonlinear volume-constrained capillary solve. The final drive is independently re-solved.
This is a repeatable local inverse algorithm, not a guarantee of reachability or global optimality.

The leading viscous acoustic wall condition and bulk forcing follow
[Bach and Bruus (2018)](https://doi.org/10.1121/1.5049579). It includes acoustic wall damping;
mean wall-layer streaming is a separate mechanism still omitted.


In [ ]:
drive = data["drive_m_s"]
assert np.array_equal(drive, np.broadcast_to(drive[0], drive.shape))
controller = np.load(campaign / "operating-state/hold-model/controller.npz")
assert not np.any(controller["gain"])
rows = np.arange(1, cfg.array_rows + 1)
phase = np.degrees(np.angle(drive[-1] * drive[-1, 0].conjugate()))
lines = ["| Row from bottom | Peak wall velocity (m/s) | Phase relative to row 1 (°) |",
         "|---:|---:|---:|"]
lines += [f"| {j} | {abs(w):.6f} | {p:.3f} |" for j,w,p in zip(rows, drive[-1], phase)]
display(Markdown("\n".join(lines)))
print("No feedback corrections. No electrical voltage calibration is available.")


## 3. Does the fluid reach and recover to that shape?

The state includes the surface and mean fluid velocity. Momentum includes inertia, viscous
stress, ALE convection, capillary/gravity traction and acoustic surface/body forces. Geometry
and acoustics are recomputed during evolution. The frozen radiation Jacobian in the implicit
step is a numerical linearization, not a controller or target-state insertion.

Four time steps reach the same held surface. Early motion differs substantially; neither a
precise settling time nor converged switch-on peak loads is claimed. Independent spatial
refinement preserves the drive and volume. The recovery cases use signed **full-aperture RMS**
perturbations of +1 µm in volume-null mode 0 and −5 µm in mode 6, both with zero initial mean
velocity. The plotted error is measured over the smaller clear pupil, so its initial RMS
need not equal the full-aperture perturbation amplitude.


In [ ]:
display(Image(filename=str(campaign / "verification/maintenance.png"), width=1000))
lines = ["| Run | Step (ms) | Final ray RMS (µm) | Max relative volume error |",
         "|---|---:|---:|---:|"]
for row in audit["runs"]:
    lines.append(f"| {row['case']} | {row['step_s'] * 1000:g} | {row['final_ray_rms_m'] * 1e6:.5f} | {row['maximum_relative_volume_error']:.2e} |")
display(Markdown("\n".join(lines)))


## 4. Optical meaning and physical limitations

The ray calculation uses the actual final slope and a fixed image plane. OPD denotes optical
path difference after subtracting piston. These are geometric-optics metrics of the mean
surface with constant refractive index, not experimental diffraction spots. A small geometric
ray radius does not imply an arbitrarily small physical optical spot.

The current holding field dissipates about 0.261 W: 0.065 W in bulk absorption and 0.196 W in
viscous acoustic wall layers. This is an energy balance, not a temperature prediction.
[Thermoviscous theory by Joergensen and Bruus (2021)](https://doi.org/10.1121/10.0005005)
shows why temperature-dependent properties and streaming must be considered before sustained
experimental claims. The model also predicts tens of nanometres of fast free-surface motion;
those oscillations and acoustic modulation of the optical index are not included in the rays.

Missing experimental inputs include sound speed/attenuation, high-frequency constitutive
response, thermal properties/boundary conditions and loaded voltage-to-velocity calibration.
Mean wall streaming, 3D perturbations, gas/cavitation physics and curing remain separate model
extensions. A continuously held liquid shape is not a retained shape after the sound is removed.


In [ ]:
optics = trace_surface(space, data["coefficients"][-1])
print(f"Geometric ray RMS at requested plane: {optics['rms_spot_at_target_m'] * 1e6:.5f} µm")
print(f"OPD RMS: {optics['opd_rms_m'] * 1e9:.4f} nm")
print(f"Best-fit sphere, freely refocused: {report['spherical_reference']['rms_spot_at_best_focus_m'] * 1e6:.3f} µm ray RMS")
print("Maximum relative acoustic power-balance error:", audit['power_balance_relative_error'])
print("Remaining physics:", ", ".join(audit['missing_physics']))


## Inspect the apparatus and provenance

Open the [offline time viewer](../artifacts/studies/S01-single-interface/cartesian-maintenance-2026-09-06/step-formation-dt00125/viewer/index.html)
or run `uv run lenslab view artifacts/studies/S01-single-interface/cartesian-maintenance-2026-09-06/step-formation-dt00125`.
Browser verification exercises real geometry/pressure changes, time playback, all cameras and
all layer controls. Each current numerical execution archives its executable sources before
iteration. The audit stores trajectory hashes, which identify local data without implying
experimental validation.
